In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import re
from pathlib import Path
from fastcore.all import L, fdelegates
from pdf_oxide import PdfDocument

## One PDF, four things that go wrong

pdf-oxide turns a page into markdown. but, there are some issues

| what goes wrong | the fix |
|----|----|
| a word broken across lines by a hyphen | `clean_md` rejoins it |
| a two-column table reflowed into one stream | `fix_layout` swaps in the plain-text layer |
| a scanned page with no text layer | `pdf_md` runs OCR |
| a web print-to-PDF with reader chrome in it | `clean_md` drops it |

`pdf_md` does all four. The rest are exported so you can run one on its own.

In [ ]:
#| export
def pdf_doc(pdf:str|Path|bytes|PdfDocument) -> PdfDocument:
    'A `PdfDocument` from a path, bytes or one you already have.'
    if isinstance(pdf, PdfDocument): return pdf
    return PdfDocument.from_bytes(pdf) if isinstance(pdf, bytes) else PdfDocument(str(pdf))

## Cleaning

`clean_md` fixes what conversion does to running text, then collapses the whitespace conversion
left behind. Pass `preserve_layout=True` to keep whitespace.

In [ ]:
#| export
_CHROME = re.compile(r'Press enter or click to view [^\n]+\n?', re.I)

def clean_md(md:str,                       # markdown from a PDF page
             preserve_layout:bool=False    # keep indentation and space runs, which a table needs
) -> str:
    'Rejoin hyphenated words, drop reader chrome, collapse the whitespace conversion leaves behind.'
    md = _CHROME.sub('', md)
    md = re.sub(r'([a-z])-\n([a-z])', r'\1\2', md)    # "docu-\nment" is one word
    md = re.sub(r'([a-z])-\n([A-Z])', r'\1-\2', md)   # "co-\nOperative" keeps its hyphen
    if not preserve_layout:
        md = re.sub(r'^ +', '', md, flags=re.M)
        md = re.sub(r' {2,}', ' ', md)
    return re.sub(r'\n{3,}', '\n\n', md)

In [ ]:
from pdflite.core import clean_md

assert clean_md('a docu-\nment') == 'a document'                  # rejoined
assert clean_md('a co-\nOperative') == 'a co-Operative'           # kept, the capital says it is real
assert clean_md('Press enter or click to view image\nreal text') == 'real text'
assert clean_md('a    b\n\n\n\nc') == 'a b\n\nc'                  # runs and blank lines collapse
assert clean_md('  a    b', preserve_layout=True) == '  a    b'   # untouched, the spaces are columns

## Scrambled layout

A two-column statement reflowed into one stream leaves its numbers stranded on their own lines,
with the labels somewhere above. `orphan_vals` counts those lines. When there are any,
`fix_layout` tries the plain-text layer instead and keeps it only if it strands fewer.

In [ ]:
#| export
_ORPHAN = re.compile(r'^[\$\-\s]*[\d,]+\.\d{2}\s*$', flags=re.M)

def orphan_vals(md:str) -> int:
    'Lines holding a money value and nothing else, which is what a reflowed column leaves behind.'
    return len(_ORPHAN.findall(md))

def scrambled_layout(md:str) -> bool:
    'True when the text layer is fine but conversion cut values loose from their labels.'
    return orphan_vals(md) > 0

def needs_ocr(md:str) -> bool:
    'True when pdf-oxide flagged the page as scanned, so there is no text layer to read.'
    return '> [OCR REQUIRED' in md.strip()

def fix_layout(pdf:PdfDocument,             # the document the page came from
               page:int,                    # 0-based page number
               md:str,                      # that page's markdown
               preserve_layout:bool=False
) -> str:
    'Swap a scrambled page for its plain-text layer, but only if that strands fewer values.'
    if not scrambled_layout(md): return md
    txt = clean_md(pdf.to_plain_text(page, preserve_layout), preserve_layout)
    return txt if txt.strip() and orphan_vals(txt) < orphan_vals(md) else md

In [ ]:
from pdflite.core import orphan_vals, scrambled_layout, needs_ocr

good = 'Opening balance   1,240.00\nFees   18.50'
bad  = 'Opening balance\nFees\n1,240.00\n18.50'
assert orphan_vals(good) == 0 and not scrambled_layout(good)
assert orphan_vals(bad) == 2 and scrambled_layout(bad)
assert needs_ocr('> [OCR REQUIRED] page 3')
assert not needs_ocr('ordinary text')

In [ ]:
from pdflite.core import fix_layout

class Stub:
    def __init__(self, txt): self.txt = txt
    def to_plain_text(self, page, preserve_layout=False): return self.txt

assert fix_layout(Stub('anything'), 0, good) == good              # not scrambled, nothing to do
assert fix_layout(Stub(good), 0, bad) == clean_md(good)           # 2 orphans down to 0, so swap
assert fix_layout(Stub('1,240.00\n18.50\n9.99'), 0, bad) == bad  # 3 orphans is worse, so keep

## Reading a whole document

`pdf_md` is the one call. It converts every page, repairs the scrambled ones, and falls back to
OCR when pdf-oxide says the page was scanned. An OCR pass that comes back empty is a failure, not
a document, so the markdown is kept instead.

`pages=False` gives one string with `---` between pages. `pages=True` gives a list, one entry per
page, which is what a chunker that cites page numbers wants.

In [ ]:
#| export
def ocr_parse(pdf:str|Path|bytes|PdfDocument,   # the document to scan
              dpi:int=300,                      # 150 is readable, 300 is good
              extract_links:bool=True,
              **kw                              # forwarded to LiteParse (num_workers, ...)
):
    'Run OCR over a scanned PDF. Returns the LiteParse result, whose `.pages` carry the text.'
    try: from liteparse import LiteParse
    except ImportError as e: raise ImportError('OCR needs liteparse: pip install liteparse') from e
    doc = pdf_doc(pdf)
    return LiteParse(ocr_enabled=True, dpi=dpi, extract_links=extract_links, **kw).parse(doc.to_bytes())

def oxide_parse(pdf:str|Path|bytes|PdfDocument,
                out_path:str|Path=None,     # dir holding the image subdir; links are relative to it
                preserve_layout:bool=False,
                image_dir:str='images',     # subdir under `out_path` for extracted images
                **kw                        # forwarded to pdf-oxide
) -> tuple:
    'Every page as cleaned, layout-repaired markdown, and whether any page wants OCR.'
    doc = pdf_doc(pdf)
    imdir = Path(out_path or 'pdfs').resolve()/image_dir
    imdir.mkdir(exist_ok=True, parents=True)
    kw = dict(image_output_dir=str(imdir), include_images=True, embed_images=False, **kw)
    def page(i):
        md = clean_md(doc.to_markdown(i, preserve_layout, **kw), preserve_layout)
        return fix_layout(doc, i, md, preserve_layout).replace(f'{imdir}/', f'{image_dir}/')
    pages = [page(i) for i in range(int(doc.page_count))]
    return pages, any(needs_ocr(p) for p in pages)

@fdelegates(oxide_parse)
def pdf_md(pdf:str|Path|bytes|PdfDocument,
           out_path:str|Path=None,
           pages:bool=False,          # True for one entry per page, False for one joined string
           ocr:str='auto',            # auto | on | off. auto asks pdf-oxide whether the page is scanned
           preserve_layout:bool=False,
           **kw                       # forwarded to oxide_parse and, on the OCR path, LiteParse
) -> str|list:
    'A PDF as markdown, with layout repaired and OCR where the page was scanned.'
    md, scanned = oxide_parse(pdf, out_path, preserve_layout, **kw)
    if ocr == 'on' or (ocr == 'auto' and scanned):
        got = [clean_md(p.text, preserve_layout) for p in ocr_parse(pdf).pages]
        if ''.join(got).strip(): md = got     # an empty OCR pass is a failure, not a document
    return md if pages else '\n---\n'.join(md)

In [ ]:
#| eval: false
pdf_md('pdfs/1997_GrolnickDeciRyan.pdf', 'pdfs/')

[liteparse] extract: 0.7ms (28 pages)


[liteparse] ocr: 13400.9ms
[liteparse] project: 23.0ms
[liteparse] total: 13424.7ms


"CHAPTER 6\n\nInternalization within the Family:\nThe Self-Determination Theory Perspective\n\nWENDY S. GROLNICK, EDWARD L. DECI, and RICHARD M. RYAN\n\nThe study of familial socialization is concerned with how children acquire the motives, values, and behavior patterns that allow them to function within the larger society (Maccoby, 1984; Zigler & Child, 1973). Although the term socialization may\nconjure up a picture of powerful parents forcing standards and behaviors onto passive\nor resistant children, effective socialization requires something more than behavior in\naccord with parental demands. It involves an inner adaptation to social requirements so\nthat children not only comply with these requirements but also accept and endorse the\nadvocated values and behaviors, experiencing them as their own. Thus, although socializing agents can force children to carry out behaviors, the real goal is for children\nto carry them out volitionally. Whereas socializing agents can “teach” thei

In [ ]:
#| eval: false
pdf_md('pdfs/attention_is_all_you_need.pdf', 'pdfs/att')

Dictionary used where Stream expected, treating as empty stream


Dictionary used where Stream expected, treating as empty stream


Dictionary used where Stream expected, treating as empty stream


Dictionary used where Stream expected, treating as empty stream


'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.\n\n## Attention Is All You Need\n\n**Ashish** **Vaswani**\n\n**Noam** **Shazeer**\n\n**Niki** **Parmar**\n\n**Jakob** **Uszkoreit** Google Brain\n\nGoogle Brain\n\nGoogle Research Google Research [avaswani@google.com](mailto:avaswani@google.com) [noam@google.com](mailto:noam@google.com) [nikip@google.com](mailto:nikip@google.com) [usz@google.com](mailto:usz@google.com)\n\n**Llion** **Jones**\n\n**Aidan** **N.** **Gomez**∗ †\n\n**Łukasz** **Kaiser** Google Research\n\nUniversity of Toronto\n\nGoogle Brain [lukaszkaiser@google.com](mailto:lukaszkaiser@google.com)\n\n[llion@google.com](mailto:llion@google.com) [aidan@cs.toronto.edu](mailto:aidan@cs.toronto.edu)\n\n**Illia** **Polosukhin**∗ ‡ [illia.polosukhin@gmail.com](mailto:illia.polosukhin@gmail.com)\n\n#### Abstract\n\nThe dominant sequence transduction models a

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()